# Regulatory Web Scraper - Orchestrator
**Description:** scheduled orchestrator to scrape regulatory bodies using Playwright and Chromium. 
**Architecture:** Single Node Job Cluster (DBR 14.3 LTS) to Unity Catalog External Volumes to Delta Tracking Tables.


In [0]:
import os
import hashlib
from datetime import datetime
from scrapers.mas_scraper import scrape_mas

# Storage and directory configuration
CATALOG = "ai_platform_dev"
SCHEMA = "regulatory_scraping"
RAW_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
CONFIG_PATH = "/Workspace/Repos/dipti@deloittetestsggmail.onmicrosoft.com/regulatory_scraper/config/mas_config.json"

os.makedirs(f"{RAW_VOLUME_PATH}/MAS", exist_ok=True)
os.makedirs(f"{RAW_VOLUME_PATH}/PDPC", exist_ok=True)

RUN_ID = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
START_TIME = datetime.now()

### Database Helper Functions

In [0]:
def get_file_hash(file_path):
    sha256_hash = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except Exception:
        return None

def check_if_processed(content_hash):
    count = spark.sql(f"""
        SELECT count(1) FROM {CATALOG}.{SCHEMA}.scraping_audit_log 
        WHERE content_hash_sha256 = '{content_hash}'
    """).collect()[0][0]
    return count > 0

def log_audit_record(
    run_id, regulator_id, start_time_sgt, end_time_sgt, execution_status,
    docs_found=0, docs_processed=0, docs_skipped=0, docs_failed=0,
    source_url=None, document_url=None, file_name=None, file_size_bytes=None,
    content_hash=None, processing_status=None, error_details=None
):
    query = f"""
        INSERT INTO {CATALOG}.{SCHEMA}.scraping_audit_log (
            run_id, regulator_id, start_time_sgt, end_time_sgt, execution_status,
            documents_found, documents_processed, documents_skipped, documents_failed,
            source_url, document_url, file_name, file_size_bytes, content_hash_sha256,
            processing_status, error_details
        ) VALUES (
            :run_id, :regulator_id, :start_time, :end_time, :exec_status,
            :found, :processed, :skipped, :failed,
            :src_url, :doc_url, :fname, :fsize, :hash,
            :proc_status, :error
        )
    """
    spark.sql(query, args={
        "run_id": run_id, "regulator_id": regulator_id, "start_time": start_time_sgt,
        "end_time": end_time_sgt, "exec_status": execution_status, "found": docs_found,
        "processed": docs_processed, "skipped": docs_skipped, "failed": docs_failed,
        "src_url": source_url, "doc_url": document_url, "fname": file_name,
        "fsize": file_size_bytes, "hash": content_hash, "proc_status": processing_status,
        "error": error_details
    })

### Main Orchestrator Trigger

In [0]:
print(f"--- Launching Scheduled Scraper Run: {RUN_ID} ---")

# Trigger MAS async pipeline
mas_docs = await scrape_mas(
    config_path=CONFIG_PATH, 
    raw_volume_path=RAW_VOLUME_PATH, 
    run_id=RUN_ID, 
    start_time=START_TIME, 
    check_processed_fn=check_if_processed, 
    log_audit_fn=log_audit_record, 
    get_hash_fn=get_file_hash
)

print(f"--- MAS Ingestion Complete: {mas_docs} documents processed ---")

In [0]:
%sql
SELECT 
    run_id, regulator_id, start_time_sgt, execution_status, 
    documents_found, documents_processed, documents_skipped, documents_failed
FROM ai_platform_dev.regulatory_scraping.scraping_audit_log 
ORDER BY start_time_sgt DESC 
LIMIT 10;

In [0]:
%pip show playwright

In [0]:
from playwright.async_api import async_playwright

async def check_chromium_version():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        print(f"The cluster is running Chromium version: {browser.version}")
        await browser.close()

await check_chromium_version()